#**𝐒𝐌𝐒 𝐒𝐩𝐚𝐦 𝐃𝐞𝐭𝐞𝐜𝐭𝐢𝐨𝐧 𝐔𝐬𝐢𝐧𝐠 𝐓𝐞𝐧𝐬𝐨𝐫𝐅𝐥𝐨𝐰 𝐇𝐮𝐛**

In [ ]:
!pip install --upgrade tensorflow tensorflow-hub tensorflow-datasets pandas scikit-learn

  Using cached tensorflow-2.21.0-cp313-cp313-manylinux_2_27_x86_64.whl.metadata (4.4 kB)


In [ ]:
!pip install numpy
!pip install tensorflow
!pip install tensorflow-hub
!pip install tensorflow-datasets
!pip install pandas
!pip install scikit-learn

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_datasets as tfds

In [ ]:
df = pd.read_csv('Spam_SMS.csv')

In [ ]:
df['Class'] = df['Class'].apply(lambda x: 1 if x == 'spam' else 0)

In [ ]:
print(df['Class'].unique())
assert df['Class'].sum() > 0
print(df['Class'].value_counts(normalize=True))

[0 1]
Class
0    0.865985
1    0.134015
Name: proportion, dtype: float64


In [ ]:
features = df['Message'].values
labels = df['Class'].values

In [ ]:
# Converted the features and labels into standard NumPy arrays to prevent indexing issues arising from PyArrow-backed data types.
features_for_split = np.asarray(features)
labels_for_split = np.asarray(labels)

train_features, temp_features, train_labels, temp_labels = train_test_split(features_for_split, labels_for_split, test_size=0.4, random_state=42, stratify=labels_for_split)

In [ ]:
validation_features, test_features, validation_labels, test_labels = train_test_split(temp_features, temp_labels, test_size=0.5, random_state=42, stratify=temp_labels)

In [ ]:
train_data = tf.data.Dataset.from_tensor_slices((train_features, train_labels))
validation_data = tf.data.Dataset.from_tensor_slices((validation_features, validation_labels))
test_data = tf.data.Dataset.from_tensor_slices((test_features, test_labels))

In [ ]:
print(f"Training samples: {len(train_features)}")
print(f"Validation samples: {len(validation_features)}")
print(f"Test samples: {len(test_features)}")

Training samples: 3344
Validation samples: 1115
Test samples: 1115


In [ ]:
train_data

<_TensorSliceDataset element_spec=(TensorSpec(shape=(), dtype=tf.string, name=None), TensorSpec(shape=(), dtype=tf.int64, name=None))>

In [ ]:
print("GPU is", "available" if tf.config.experimental.list_physical_devices("GPU") else "NOT AVAILABLE")

GPU is NOT AVAILABLE


In [ ]:
train_example_batch, train_labels_batch = next(iter(train_data.batch(10)))

In [ ]:
train_example_batch

<tf.Tensor: shape=(10,), dtype=string, numpy=
array([b'I will come tomorrow di',
       b"sure, but make sure he knows we ain't smokin yet",
       b'SERIOUSLY. TELL HER THOSE EXACT WORDS RIGHT NOW.',
       b'RGENT! This is the 2nd attempt to contact U!U have WON \xc2\xa31250 CALL 09071512433 b4 050703 T&CsBCM4235WC1N3XX. callcost 150ppm mobilesvary. max\xc2\xa37. 50',
       b'Imagine you finally get to sink into that bath after I have put you through your paces, maybe even having you eat me for a while before I left ... But also imagine the feel of that cage on your cock surrounded by the bath water, reminding you always who owns you ... Enjoy, my cuck',
       b'So when you gonna get rimac access', b"Can... I'm free...",
       b'Hey... are you going to quit soon? Xuhui and i working till end of the month',
       b'Oh god..taken the teeth?is it paining',
       b'Jane babes not goin 2 wrk, feel ill after lst nite. Foned in already cover 4 me chuck.:-)'],
      dtype=object)>

In [ ]:
train_labels_batch

<tf.Tensor: shape=(10,), dtype=int64, numpy=array([0, 0, 0, 1, 0, 0, 0, 0, 0, 0])>

In [ ]:
embedding= "https://tfhub.dev/google/nnlm-en-dim50/2"
hub_layer = hub.KerasLayer(embedding, input_shape=[], dtype=tf.string, trainable=True)

In [ ]:
hub_layer(train_example_batch[:3])

<tf.Tensor: shape=(3, 50), dtype=float32, numpy=
array([[-0.06670409,  0.07737008, -0.35333452,  0.22637588,  0.09409093,
         0.29273608,  0.03913876,  0.23777115, -0.02561577,  0.42069697,
        -0.11204858,  0.15666689,  0.17974742,  0.02520393,  0.04589377,
        -0.085166  , -0.06564827,  0.0456499 ,  0.05225061,  0.12376789,
        -0.03745144, -0.14682987, -0.12409767, -0.10252394,  0.10929826,
         0.07947893, -0.3323373 , -0.07276095, -0.1559159 ,  0.00283341,
        -0.10405201,  0.37710106,  0.09762429,  0.06715084, -0.1703025 ,
         0.35511464,  0.08389769,  0.1706751 ,  0.05141801, -0.17938219,
        -0.05126514,  0.08035786,  0.05339505, -0.02477965, -0.06060382,
         0.2850946 , -0.2771498 , -0.11974315,  0.06855694,  0.21343032],
       [ 0.25445944, -0.04932294, -0.15551217,  0.23360777, -0.12458177,
        -0.12338907, -0.10359492,  0.06022901, -0.45820597,  0.21423243,
         0.10693872, -0.03889309,  0.01361192,  0.24119245, -0.32566535,
 

#**Creating the Sequential Model**

Encountered a ValueError while building the model in Google Colab, caused by a version incompatibility between the installed TensorFlow and TensorFlow Hub packages.

The root cause was that tensorflow_hub.keras_layer.KerasLayer wasn't being added to the Sequential model correctly. I resolved this by wrapping the hub.KerasLayer instance inside a tf.keras.layers.Lambda layer, making the Hub layer compatible with the Sequential architecture in this Keras version. After applying the fix, model summary executed successfully, confirming the architecture was built correctly.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Lambda(lambda x: hub_layer(x), dtype=tf.string),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1)
])
model.build((None,))

In [ ]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lambda_2 (Lambda)               │ (None, 50)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │           816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 833 (3.25 KB)

 Trainable params: 833 (3.25 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(optimizer='adam', loss=tf.keras.losses.BinaryCrossentropy(from_logits=True), metrics=['accuracy'])

In [ ]:
history = model.fit(train_data.shuffle(10000).batch(100), epochs=75, validation_data=validation_data.batch(100), verbose=1)

Epoch 1/75
34/34 ━━━━━━━━━━━━━━━━━━━━ 4s 33ms/step - accuracy: 0.8660 - loss: 0.6339 - val_accuracy: 0.8655 - val_loss: 0.5687
Epoch 2/75
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.8660 - loss: 0.4935 - val_accuracy: 0.8655 - val_loss: 0.4316
Epoch 3/75
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.8660 - loss: 0.3849 - val_accuracy: 0.8655 - val_loss: 0.3631
Epoch 4/75
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.8660 - loss: 0.3383 - val_accuracy: 0.8655 - val_loss: 0.3324
Epoch 5/75
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.8660 - loss: 0.3139 - val_accuracy: 0.8655 - val_loss: 0.3097
Epoch 6/75
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.8660 - loss: 0.2934 - val_accuracy: 0.8655 - val_loss: 0.2898
Epoch 7/75
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.8660 - loss: 0.2752 - val_accuracy: 0.8682 - val_loss: 0.2710
Epoch 8/75
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.8687 - loss: 0.2586 - val_accuracy: 0.8744 - v

In [ ]:
results = model.evaluate(test_data.batch(100), verbose=2)

for name, value in zip(model.metrics_names, results):
  print("%s: %.3f" % (name, value))

12/12 - 0s - 4ms/step - accuracy: 0.9345 - loss: 0.1705
loss: 0.170
compile_metrics: 0.935
